# Capítulo 6 — EDA asistido y taller integrador

**Serie Ciencia de Datos · Tomo 2 — Herramientas Python para Data Science**  
José Fernando Meza

Código del capítulo, extraído del libro. Repositorio: https://github.com/josefer76/serie-ciencia-de-datos

> Ejecute las celdas en orden: algunas dependen de variables definidas antes.

## Código del capítulo

### Introducción del capítulo

In [ ]:
!pip install -q pandas==2.3.3 ydata-profiling==4.18.4

### 6.2 Perfilado automático con ydata-profiling: qué entrega y qué no

In [ ]:
import pandas as pd
from ydata_profiling import ProfileReport

DATOS = ("https://raw.githubusercontent.com/josefer76/"
         "serie-ciencia-de-datos/main/libro2/datos/")
datos = pd.read_csv(DATOS + "pedidos_taller.csv", parse_dates=["fecha"])

*Salida en el libro:*

```text
DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport
```

In [ ]:
perfil = ProfileReport(datos, title="Perfilado del dataset",
                       progress_bar=False)
perfil.to_file("reporte.html")

In [ ]:
descripcion = perfil.get_description()
for alerta in descripcion.alerts:
    print(alerta)

*Salida en el libro:*

```text
Dataset has 60 (1.2%) duplicate rows
[categoria] is highly overall correlated with [precio_unitario]
[precio_unitario] is highly overall correlated with [categoria]
[precio_unitario] 304 (6.0%) missing values
[costo_envio] 606 (12.0%) missing values
[id_pedido] is uniformly distributed
```

In [ ]:
variables = descripcion.variables
print("mínimo de precio_unitario:", variables["precio_unitario"]["min"])
print("valores distintos de region:", variables["region"]["n_distinct"])
print(sorted(variables["region"]["value_counts_without_nan"].index))
print("precios negativos:", (datos["precio_unitario"] < 0).sum())

*Salida en el libro:*

```text
mínimo de precio_unitario: -301.82
valores distintos de region: 7
['Este', 'NORTE ', 'Norte', 'Oeste', 'Sur', 'norte', 'sur']
precios negativos: 7
```

In [ ]:
# Rápido, pero SIN detección de duplicados ni correlaciones
rapido = ProfileReport(datos, minimal=True, progress_bar=False)

# Completo (el valor por defecto): más lento, pero detecta ambas cosas
completo = ProfileReport(datos, minimal=False, progress_bar=False)

for nombre, p in [("minimal=True: ", rapido), ("minimal=False:", completo)]:
    print("duplicados con", nombre,
          p.get_description().table.get("n_duplicates"))

*Salida en el libro:*

```text
duplicados con minimal=True:  None
duplicados con minimal=False: 60
```

### 6.3 IA aplicada al análisis de datos: auditoría, verificación y privacidad

#### El riesgo específico: código que corre y resultados que mienten

In [ ]:
import pandas as pd
import numpy as np

sucursales = pd.DataFrame({
    "sucursal": ["Central"] * 950 + ["Kiosco"] * 50,
    "ticket": list(np.full(950, 100.0)) + list(np.full(50, 1000.0)),
})

# Versión generada: promedia los promedios de cada grupo
incorrecta = sucursales.groupby("sucursal")["ticket"].mean().mean()

# Versión correcta: promedio sobre todas las observaciones
correcta = sucursales["ticket"].mean()

print(f"{incorrecta:.2f}")   # 550.00
print(f"{correcta:.2f}")     # 145.00

### 6.4 Taller integrador: EDA completo de extremo a extremo

#### Fase 1 — Reconocimiento

In [ ]:
import pandas as pd
import numpy as np

DATOS = ("https://raw.githubusercontent.com/josefer76/"
         "serie-ciencia-de-datos/main/libro2/datos/")
datos = pd.read_csv(DATOS + "pedidos_taller.csv", parse_dates=["fecha"])

print(datos.shape)                                   # (5060, 7)
print(datos.dtypes)
print((datos.isna().mean() * 100).round(2))          # faltantes en %
print("duplicados exactos:", datos.duplicated().sum())
print("cardinalidad de region:", datos["region"].nunique())
print(sorted(datos["region"].unique()))
print(datos[["unidades", "precio_unitario", "costo_envio"]]
      .describe().round(2))

*Salida en el libro:*

```text
(5060, 7)
id_pedido                   int64
fecha              datetime64[ns]
region                     object
categoria                  object
unidades                    int64
precio_unitario           float64
costo_envio               float64
dtype: object
id_pedido           0.00
fecha               0.00
region              0.00
categoria           0.00
unidades            0.00
precio_unitario     6.01
costo_envio        11.98
dtype: float64
duplicados exactos: 60
cardinalidad de region: 7
['Este', 'NORTE ', 'Norte', 'Oeste', 'Sur', 'norte', 'sur']
       unidades  precio_unitario  costo_envio
count   5060.00          4756.00      4454.00
mean      14.75           134.65        12.73
std       41.28            93.24         7.15
min        1.00          -301.82         0.01
25%        7.00            61.48         6.52
50%       12.00           105.64        12.96
75%       19.00           196.56        18.87
max      888.00           460.71        25.00
```

#### Fase 2 — Limpieza, con registro de decisiones

In [ ]:
limpio = datos.copy()

# 1. Duplicados exactos: 60 filas idénticas, casi con seguridad error de
# carga.
limpio = limpio.drop_duplicates()

# 2. Normalización de categorías: 7 variantes de escritura -> 4 regiones
# reales.
limpio["region"] = limpio["region"].str.strip().str.lower().str.capitalize()

# 3. Precios negativos: imposibles en el dominio. Se convierten a NaN en lugar
# de borrarlos, para tratarlos junto al resto de faltantes con un criterio
# único.
limpio.loc[limpio["precio_unitario"] < 0, "precio_unitario"] = np.nan

# 4. Unidades implausibles: REGLA DE DOMINIO, no estadística.
LIMITE_PLAUSIBLE_UNIDADES = 100
limpio = limpio[limpio["unidades"] <= LIMITE_PLAUSIBLE_UNIDADES]

# 5. Precio faltante: es la variable central del análisis; imputarla
# inventaría el dato que se quiere medir. Se eliminan esas filas.
limpio = limpio.dropna(subset=["precio_unitario"])

# 6. Costo de envío faltante: variable secundaria, con distribución acotada.
#    Se imputa con la mediana, robusta frente a extremos.
limpio["costo_envio"] = limpio["costo_envio"].fillna(
    limpio["costo_envio"].median())

print(f"filas: {len(datos)} -> {len(limpio)} "
      f"({len(limpio) / len(datos):.1%})")

*Salida en el libro:*

```text
filas: 5060 -> 4681 (92.5%)
```

#### Fase 3 — Ingeniería de variables y optimización

In [ ]:
# vectorizado
limpio["ingreso"] = limpio["precio_unitario"] * limpio["unidades"]
limpio["ingreso_neto"] = limpio["ingreso"] - limpio["costo_envio"]
# Nótese "M" y no "ME": los periodos conservan el alias antiguo (Sección 4.8)
limpio["mes"] = limpio["fecha"].dt.to_period("M").astype(str)
limpio["dia_semana"] = limpio["fecha"].dt.day_name()

# Segmentación con np.select, tratando el faltante explícitamente (Sección
# 4.3)
limpio["segmento"] = np.select(
    [limpio["ingreso"].isna(), limpio["ingreso"] > 2000,
     limpio["ingreso"] > 800],
    ["Sin dato", "Alto", "Medio"], default="Bajo",
)

# Optimización de tipos (Secciones 3.9 y 4.2)
antes = limpio.memory_usage(deep=True).sum()
for columna in ["region", "categoria", "segmento", "dia_semana"]:
    limpio[columna] = limpio[columna].astype("category")
limpio["unidades"] = pd.to_numeric(limpio["unidades"], downcast="unsigned")
despues = limpio.memory_usage(deep=True).sum()
print(f"memoria: {antes / 1024**2:.2f} MB -> {despues / 1024**2:.2f} MB "
      f"(ahorro {1 - despues / antes:.1%})")

*Salida en el libro:*

```text
memoria: 1.55 MB -> 0.52 MB (ahorro 66.2%)
```

#### Fase 4 — Análisis

In [ ]:
resumen_categoria = limpio.groupby("categoria", observed=True).agg(
    ingreso_total=("ingreso", "sum"),
    ticket_promedio=("ingreso", "mean"),
    unidades=("unidades", "sum"),
).round(2).sort_values("ingreso_total", ascending=False)
print(resumen_categoria)

*Salida en el libro:*

```text
             ingreso_total  ticket_promedio  unidades
categoria                                            
Electrónica     4445988.84          3157.66     17730
Hogar           1615970.15          1185.60     16957
Deportes        1142146.92          1615.48      8891
Ropa             733052.54           609.35     15160
```

#### Fase 5 — Visualización y síntesis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
serie_mensual = limpio.set_index("fecha")["ingreso"].resample("ME").sum()
correlaciones = limpio[["unidades", "precio_unitario", "ingreso",
                        "costo_envio"]].corr()

fig, ejes = plt.subplots(2, 2, figsize=(11, 8))

ejes[0, 0].plot(serie_mensual.index, serie_mensual.values / 1000, marker="o",
                color="#1F3864", linewidth=1.8)
ejes[0, 0].set_title("Evolución mensual del ingreso", fontweight="bold",
                     fontsize=10)
ejes[0, 0].set_ylabel("Ingreso (miles)")
ejes[0, 0].tick_params(axis="x", rotation=45)

orden_cat = resumen_categoria.sort_values("ingreso_total").index
ejes[0, 1].barh(orden_cat,
                resumen_categoria.loc[orden_cat, "ingreso_total"] / 1000,
                color="#2E75B6")
ejes[0, 1].set_title("Ingreso por categoría", fontweight="bold", fontsize=10)
ejes[0, 1].set_xlabel("Ingreso total (miles)")

sns.boxplot(data=limpio, x="categoria", y="precio_unitario", ax=ejes[1, 0],
            hue="categoria", legend=False, palette="deep")
ejes[1, 0].set_title("Distribución de precios por categoría",
                     fontweight="bold", fontsize=10)
ejes[1, 0].tick_params(axis="x", rotation=20)

sns.heatmap(correlaciones, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ejes[1, 1],
            cbar_kws={"shrink": 0.8})
ejes[1, 1].set_title("Matriz de correlación", fontweight="bold", fontsize=10)

fig.tight_layout()
fig.savefig("fig_6_taller.png", dpi=160, bbox_inches="tight")

### Ejercicios propuestos

In [ ]:
import numpy as np
import pandas as pd

ventas = pd.DataFrame({
    "sucursal": ["Central"] * 950 + ["Kiosco"] * 50,
    "ticket": list(np.full(950, 100.0)) + list(np.full(50, 1000.0)),
})

# Objetivo declarado: calcular el ticket promedio global de la compañía
ticket_promedio = ventas.groupby("sucursal")["ticket"].mean().mean()
print(f"Ticket promedio de la compañía: {ticket_promedio:.2f}")

## Soluciones de los ejercicios

### [Ex-L2-C6-01] Perfilado automático frente a EDA manual

#### Desarrollo con código

Si ejecutó antes la sección 6.2 en esta misma sesión, las cinco primeras líneas de la salida del libro (el aviso de importación de ydata-profiling) no vuelven a aparecer: Python solo muestra ese aviso la primera vez que se importa la biblioteca.

In [ ]:
import pandas as pd
from ydata_profiling import ProfileReport

DATOS = ("https://raw.githubusercontent.com/josefer76/"
         "serie-ciencia-de-datos/main/libro2/datos/")
datos_taller = pd.read_csv(DATOS + "pedidos_taller.csv",
                           parse_dates=["fecha"])

# Dos configuraciones, deliberadamente, para comparar qué detecta cada una
perfil_minimo = ProfileReport(datos_taller, minimal=True,
                              progress_bar=False)
perfil_completo = ProfileReport(datos_taller, minimal=False,
                                progress_bar=False)
perfil_completo.to_file("reporte.html")

# Las alertas son accesibles programáticamente, no solo en el HTML
for nombre, perfil in [("minimal=True", perfil_minimo),
                       ("modo completo", perfil_completo)]:
    print(f"--- {nombre}")
    for alerta in perfil.get_description().alerts:
        print(alerta.alert_type.name, alerta.column_name)
print("duplicados (minimal=True):",
      perfil_minimo.get_description().table.get("n_duplicates"))
print("duplicados (modo completo):",
      perfil_completo.get_description().table.get("n_duplicates"))

#### Salida esperada

*Salida en el libro:*

```text
DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport
--- minimal=True
MISSING precio_unitario
MISSING costo_envio
--- modo completo
DUPLICATES None
HIGH_CORRELATION categoria
HIGH_CORRELATION precio_unitario
MISSING precio_unitario
MISSING costo_envio
UNIFORM id_pedido
duplicados (minimal=True): None
duplicados (modo completo): 60
```

In [ ]:
variables = perfil_completo.get_description().variables
print("mínimo de precio_unitario:", variables["precio_unitario"]["min"])
print("precios negativos:", (datos_taller["precio_unitario"] < 0).sum())
print("valores distintos de region:", variables["region"]["n_distinct"])
print("rango de fechas:", variables["fecha"]["range"].days, "días")

*Salida en el libro:*

```text
mínimo de precio_unitario: -301.82
precios negativos: 7
valores distintos de region: 7
rango de fechas: 729 días
```

### [Ex-L2-C6-02] Auditoría de código generado por IA

#### Desarrollo con código

In [ ]:
import pandas as pd
import numpy as np

sucursales = pd.DataFrame({
    "sucursal": ["Central"] * 950 + ["Kiosco"] * 50,
    "ticket": list(np.full(950, 100.0)) + list(np.full(50, 1000.0)),
})

# CÓDIGO GENERADO — objetivo declarado: "ticket promedio global de la
# compañía"
ticket_promedio = sucursales.groupby("sucursal")["ticket"].mean().mean()
print(f"{ticket_promedio:.2f}")     # 550.00

# CÓDIGO CORRECTO
correcto = sucursales["ticket"].mean()
print(f"{correcto:.2f}")            # 145.00   -> sobreestimación del 279 %

#### Salida esperada

*Salida en el libro:*

```text
550.00
145.00
```

In [ ]:
# Cuatro filas cuyo resultado se calcula mentalmente:
juguete = pd.DataFrame({"sucursal": ["A", "A", "A", "B"],
                        "ticket": [10.0, 10.0, 10.0, 100.0]})

# A mano: (10 + 10 + 10 + 100) / 4 = 32.5
print(juguete.groupby("sucursal")["ticket"].mean().mean())
# 55.0  -> NO coincide

*Salida en el libro:*

```text
55.0
```

In [ ]:
# Objetivo: predecir si un cliente RENOVARÁ su contrato.
generador = np.random.default_rng(7)
clientes = pd.DataFrame({
    "antiguedad_meses": generador.integers(1, 60, 1000),
    "num_productos": generador.integers(1, 5, 1000),
})
# Variable objetivo: la probabilidad de renovar crece con la antigüedad
probabilidad = 1 / (1 + np.exp(-(0.05 * clientes["antiguedad_meses"] - 1.5)))
clientes["renovo"] = (generador.random(1000) < probabilidad).astype(int)

# La variable envenenada: solo existe DESPUÉS de conocer el desenlace.
# Vale -1 si el cliente no renovó, y un número de días si sí lo hizo.
clientes["dias_hasta_renovacion"] = np.where(
    clientes["renovo"] == 1, generador.integers(1, 30, 1000), -1)

# LAS DOS SEÑALES DE ALARMA
print(round(clientes["antiguedad_meses"].corr(clientes["renovo"]), 3))
# 0.382  <- señal real
print(round(clientes["dias_hasta_renovacion"].corr(clientes["renovo"]), 3))
# 0.797  <- sospechoso

# La comprobación decisiva: predecir usando SOLO esa variable
prediccion_trivial = (clientes["dias_hasta_renovacion"] > 0).astype(int)
print((prediccion_trivial == clientes["renovo"]).mean())
# 1.0  -> 100 %

*Salida en el libro:*

```text
0.382
0.797
1.0
```

### [Ex-L2-C6-03] Proyecto final: EDA completo

#### Desarrollo con código

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine

# ---------------------------------------------
# FASE 1 — RECONOCIMIENTO
# ---------------------------------------------
wine = load_wine(as_frame=True)
vinos = wine.frame.copy()
vinos["cultivar"] = vinos["target"].map({0: "Cultivar A", 1: "Cultivar B",
                                         2: "Cultivar C"})
vinos = vinos.drop(columns="target")

print(vinos.shape)                     # (178, 14)
print(vinos.isna().sum().sum())        # 0  -> sin faltantes
print(vinos.duplicated().sum())        # 0  -> sin duplicados

# ---------------------------------------------
# FASE 2/3 — OPTIMIZACIÓN E INGENIERÍA DE VARIABLES
# ---------------------------------------------
memoria_antes = vinos.memory_usage(deep=True).sum()
optimizado = vinos.copy()
for columna in optimizado.select_dtypes("float64").columns:
    optimizado[columna] = pd.to_numeric(optimizado[columna], downcast="float")
optimizado["cultivar"] = optimizado["cultivar"].astype("category")
memoria_despues = optimizado.memory_usage(deep=True).sum()
print(f"memoria: {memoria_antes / 1024:.1f} KB -> "
      f"{memoria_despues / 1024:.1f} KB "
      f"(ahorro del {(1 - memoria_despues / memoria_antes) * 100:.1f} %)")

# Variables derivadas, en el peldaño más alto de la escalera (Sección 4.3)
# peldaño 1
optimizado["ratio_fenoles"] = (optimizado["flavanoids"] /
                               optimizado["total_phenols"]).round(3)
# peldaño 1
optimizado["intensidad_relativa"] = (optimizado["color_intensity"] /
                                     optimizado["hue"]).round(3)
# peldaño 3
optimizado["nivel_alcohol"] = np.select(
    [optimizado["alcohol"].isna(), optimizado["alcohol"] > 13.5,
     optimizado["alcohol"] > 12.5],
    ["Sin dato", "Alto", "Medio"], default="Bajo")

# ---------------------------------------------
# FASE 4 — ANÁLISIS
# ---------------------------------------------
resumen = optimizado.groupby("cultivar", observed=True).agg(
    n=("alcohol", "size"),
    alcohol_medio=("alcohol", "mean"),
    flavanoides_medio=("flavanoids", "mean"),
    prolina_media=("proline", "mean"),
).round(2)

# Selección EXPLÍCITA de columnas numéricas antes de correlacionar
variables_clave = ["alcohol", "flavanoids", "color_intensity", "proline",
                   "hue"]
correlaciones = optimizado[variables_clave].corr()

# Pearson frente a Spearman (Sección 5.6): la discrepancia es un hallazgo
pearson = optimizado["flavanoids"].corr(optimizado["color_intensity"],
                                        method="pearson")
spearman = optimizado["flavanoids"].corr(optimizado["color_intensity"],
                                         method="spearman")

print("\nPerfil por cultivar:")
print(resumen)
print("\nPares con |correlación| > 0.4:")
pares = [(a, b, correlaciones.loc[a, b])
         for i, a in enumerate(variables_clave)
         for b in variables_clave[i + 1:]]
for a, b, valor in sorted(pares, key=lambda par: -abs(par[2])):
    if abs(valor) > 0.4:
        print(f"  {a} - {b}: {valor:.3f}")
print(f"\nPearson {pearson:.3f} | Spearman {spearman:.3f} | "
      f"diferencia {abs(pearson - spearman):.3f}")
print("\nRango de alcohol por cultivar:")
print(optimizado.groupby("cultivar", observed=True)["alcohol"]
      .agg(["min", "max"]).round(2))

#### Salida esperada

*Salida en el libro:*

```text
(178, 14)
0
0
memoria: 28.5 KB -> 9.6 KB (ahorro del 66.2 %)

Perfil por cultivar:
             n  alcohol_medio  flavanoides_medio  prolina_media
cultivar                                                       
Cultivar A  59          13.74               2.98    1115.709961
Cultivar B  71          12.28               2.08     519.510010
Cultivar C  48          13.15               0.78     629.900024

Pares con |correlación| > 0.4:
  alcohol - proline: 0.644
  alcohol - color_intensity: 0.546
  flavanoids - hue: 0.543
  color_intensity - hue: -0.522
  flavanoids - proline: 0.494

Pearson -0.172 | Spearman -0.043 | diferencia 0.129

Rango de alcohol por cultivar:
              min    max
cultivar                
Cultivar A  12.85  14.83
Cultivar B  11.03  13.86
Cultivar C  12.20  14.34
```

In [ ]:
sns.set_theme(style="whitegrid")
fig, ejes = plt.subplots(2, 3, figsize=(14, 8))

sns.boxplot(data=optimizado, x="cultivar", y="alcohol", ax=ejes[0, 0],
            hue="cultivar", legend=False, palette="deep")
ejes[0, 0].set_title("V1. ¿Difiere el alcohol entre cultivares?",
                     fontweight="bold", fontsize=9)

sns.histplot(data=optimizado, x="flavanoids", hue="cultivar", kde=True,
             ax=ejes[0, 1], palette="deep")
ejes[0, 1].set_title("V2. ¿Cómo se distribuyen los flavanoides?",
                     fontweight="bold", fontsize=9)

sns.scatterplot(data=optimizado, x="flavanoids", y="color_intensity",
                hue="cultivar", ax=ejes[0, 2], palette="deep", s=28,
                alpha=0.8)
ejes[0, 2].set_title("V3. ¿Se relacionan flavanoides e intensidad?",
                     fontweight="bold", fontsize=9)

sns.heatmap(correlaciones, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ejes[1, 0],
            cbar_kws={"shrink": 0.7})
ejes[1, 0].set_title("V4. Matriz de correlación", fontweight="bold",
                     fontsize=9)
ejes[1, 0].tick_params(axis="x", rotation=40)

sns.violinplot(data=optimizado, x="cultivar", y="proline", ax=ejes[1, 1],
               hue="cultivar", legend=False, palette="deep")
ejes[1, 1].set_title("V5. ¿La prolina separa los cultivares?",
                     fontweight="bold", fontsize=9)

conteo = (optimizado.groupby(["cultivar", "nivel_alcohol"], observed=True)
          .size().unstack(fill_value=0))
conteo.plot(kind="bar", stacked=True, ax=ejes[1, 2], colormap="Blues_r")
ejes[1, 2].set_title("V6. Nivel de alcohol por cultivar", fontweight="bold",
                     fontsize=9)
ejes[1, 2].tick_params(axis="x", rotation=15)
ejes[1, 2].legend(title="Nivel", fontsize=7)

fig.suptitle("Proyecto final: EDA del dataset Wine (UCI, 178 muestras)",
             fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig("fig_sol_c6_03.png", dpi=160, bbox_inches="tight")